In [1]:
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
) 
import torch
from qwen_vl_utils import process_vision_info

/home/mayflower/.local/share/mamba/envs/EHA/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mayflower/.local/share/mamba/envs/EHA/lib/python3.10/site-packages/transformers/utils/hub.py:109: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
def load_model(model_id):
    """Load the model and processor."""
    min_pixels = 256 * 28 * 28
    max_pixels = 512 * 28 * 28
    processor = AutoProcessor.from_pretrained(
        model_id, trust_remote_code=True, min_pixels=min_pixels, max_pixels=max_pixels
    )

    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        dtype=torch.bfloat16,
        trust_remote_code=True,
        #attn_implementation="flash_attention_2",
        device_map="auto",
    )

    model.eval()
    return model, processor

In [10]:
def prepare_inputs(model, processor, image_paths, questions):
    # Ensure inputs are lists for batch processing
    if isinstance(image_paths, str):
        image_paths = [image_paths]
    if isinstance(questions, str):
        questions = [questions]
    
    # Validate batch sizes match
    batch_size = len(image_paths)
    if len(questions) != batch_size:
        raise ValueError(f"Batch size mismatch: {len(image_paths)} images vs {len(questions)} questions")
    
    # Build messages for each item in the batch
    messages = []
    for img_path, question in zip(image_paths, questions):
        messages.append({
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": question},
            ],
        })
    
    # Create one text + image pair per sample
    # Apply chat template
    texts = [
        processor.apply_chat_template([m], tokenize=False, add_generation_prompt=True)
        for m in messages
    ]
    print(f"[DEBUG] processor received: {len(texts)} texts, {len(image_paths)} images")
    # 🚀 Instead of process_vision_info → pass image_paths directly
    inputs = processor(
        text=texts,
        images=image_paths,    # let processor handle batching
        padding=True,
        return_tensors="pt",
    )

    return inputs.to(model.device)

In [5]:
model_id = "google/gemma-3-4b-it"
model, processor = load_model(model_id)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.56s/it]


In [11]:
# Batch input
inputs = prepare_inputs(
    model, 
    processor, 
    [["../datasets/coco2014/val2014/COCO_val2014_000000000042.jpg"], ["../datasets/coco2014/val2014/COCO_val2014_000000000073.jpg"], ["../datasets/coco2014/val2014/COCO_val2014_000000000074.jpg"]],
    ["Describe this", "What color is this?", "Count the objects"]
)

[DEBUG] processor received: 3 texts, 3 images


In [12]:
print(inputs)

{'input_ids': tensor([[     0,      0,      0,      2,      2,    105,   2364,    109, 255999,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144, 262144,
         26214

In [13]:
# Then generate
outputs = model.generate(**inputs, max_new_tokens=32)
responses = processor.batch_decode(outputs, skip_special_tokens=True)

In [14]:
print(responses)

["user\n\n\n\n\nDescribe this\nmodel\nHere's a description of the image:\n\n**Overall Impression:**\n\nThe image shows a fluffy, brown dog comfortably nestled inside a laundry basket overflowing with shoes", 'user\n\n\n\n\nWhat color is this?\nmodel\nBased on the image, the motorcycle is primarily **light green** with black accents. \n\nYou can see this in the tank, side panels, and some', "user\n\n\n\n\nCount the objects\nmodel\nHere's a breakdown of the objects I can identify in the image:\n\n*   **Dog:** 1\n*   **Bike:** 1\n"]


In [ ]:
video_inputs = []
video_input is None
if video_inputs is not None:
            video_inputs.extend(video_input)